# Maze Evaluation Visualization

This notebook visualizes and analyzes the saved output of:

```bash
python evaluate_trained_model.py \
  --checkpoint checkpoints/URM-maze \
  --max_problems 4096 \
  --loops 32 \
  --batch_size 1024 \
  --hidden_diff_threshold 0.1
```

It first loads the aggregate metrics already saved under `checkpoints/URM-maze/`, then optionally runs and caches a more detailed pass so we can inspect per-maze predictions, final stop steps, and representative failures.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

import maze_visualization_utils as maze_viz

REPO_ROOT = maze_viz.resolve_repo_root()
maze_viz.ensure_repo_root_on_syspath(REPO_ROOT)
maze_viz.set_plot_style()

PATHS = maze_viz.default_paths(REPO_ROOT)
CHECKPOINT_DIR = PATHS['checkpoint_dir']
METRICS_PATH = PATHS['metrics_path']
DETAILED_CACHE_PATH = PATHS['detailed_path']

SPLIT = 'test'
BATCH_SIZE = 1024
MAX_PROBLEMS = 4096
LOOPS = 32
HIDDEN_DIFF_THRESHOLD = 0.1
FORCE_RERUN_DETAILED = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'repo_root            : {REPO_ROOT}')
print(f'checkpoint_dir       : {CHECKPOINT_DIR}')
print(f'metrics_path         : {METRICS_PATH}')
print(f'detailed_cache_path  : {DETAILED_CACHE_PATH}')
print(f'device               : {DEVICE}')

In [ ]:
metrics_payload = maze_viz.load_payload(METRICS_PATH)

display(Markdown('## Aggregate Metrics'))
display(Markdown(f'```bash\n{maze_viz.COMMAND}\n```'))
print(maze_viz.format_metrics_table(metrics_payload))

loop_rows = maze_viz.build_loop_summary(metrics_payload)
if loop_rows:
    print()
    print(f"{'loop':>6} {'token_acc':>12} {'exact_acc':>12} {'lm_loss':>12} {'avg_steps':>12}")
    print('-' * 60)
    for row in loop_rows:
        print(
            f"{int(row['loop']):>6d} "
            f"{row['accuracy']:>12.6f} "
            f"{row['exact_accuracy']:>12.6f} "
            f"{row['lm_loss']:>12.6f} "
            f"{row['steps']:>12.6f}"
        )

fig, _axes = maze_viz.plot_loop_metrics(metrics_payload)
plt.show()

In [ ]:
detailed_payload = maze_viz.load_or_run_detailed_evaluation(
    checkpoint_dir=CHECKPOINT_DIR,
    detailed_cache_path=DETAILED_CACHE_PATH,
    split=SPLIT,
    batch_size=BATCH_SIZE,
    max_problems=MAX_PROBLEMS,
    loops=LOOPS,
    hidden_diff_threshold=HIDDEN_DIFF_THRESHOLD,
    device=DEVICE,
    force_rerun=FORCE_RERUN_DETAILED,
)

stats = maze_viz.compute_example_stats(detailed_payload)

print(f"processed_problems: {detailed_payload['processed_problems']}")
print(f"cached detailed file: {DETAILED_CACHE_PATH.exists()}")
print(f"outputs available for sets: {list(detailed_payload['outputs'].keys())}")

In [ ]:
analysis_lines = maze_viz.build_analysis_lines(detailed_payload, stats)
display(Markdown('## Key Findings\n' + '\n'.join(f'- {line}' for line in analysis_lines)))

In [ ]:
display(Markdown('## Runtime Diagnostics'))
fig, _axes = maze_viz.plot_runtime_diagnostics(stats, detailed_payload)
plt.show()

display(Markdown('## Path-Centric Distributions'))
fig, _axes, error_summary = maze_viz.plot_path_distributions(stats)
plt.show()

display(Markdown(
    '## Error Summary\n'
    f"- solved mazes: {error_summary['solved_missed_mean']:.2f} missed path cells, {error_summary['solved_extra_mean']:.2f} extra path cells on average\n"
    f"- unsolved mazes: {error_summary['unsolved_missed_mean']:.2f} missed path cells, {error_summary['unsolved_extra_mean']:.2f} extra path cells on average"
))

In [ ]:
display(Markdown('## Legends'))
fig, _axes = maze_viz.plot_legends()
plt.show()

In [ ]:
display(Markdown('## Fast Solved Examples'))
fast_solved = np.flatnonzero(stats['exact'] & (stats['steps'] == stats['steps'].min()))[:4]
maze_viz.show_examples(stats, fast_solved, 'Fast solved examples')

In [ ]:
display(Markdown('## Slow or Late Solved Examples'))
slow_solved = np.flatnonzero(stats['exact'] & (stats['steps'] >= 4))
if slow_solved.size == 0:
    slow_solved = np.flatnonzero(stats['exact'] & (stats['steps'] == 3))
slow_solved = slow_solved[:4]
maze_viz.show_examples(stats, slow_solved, 'Slow or late solved examples')

In [ ]:
display(Markdown('## Hard Failures'))
failure_order = np.argsort(stats['path_f1'])
hard_failures = [int(idx) for idx in failure_order if not stats['exact'][idx]][:4]
maze_viz.show_examples(stats, hard_failures, 'Hard failures ranked by lowest path F1')

## Notes

- `exact accuracy` is the strict 900-token match used by `evaluate_trained_model.py`.
- `path F1` only looks at the predicted `o` cells, so it is more informative than token accuracy for maze behavior.
- `final step` is the step at which an example stopped after applying the `hidden_diff_threshold=0.1` pruning rule.